Memory

In [2]:
# 初始化配置
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

import warnings
warnings.filterwarnings("ignore")

In [5]:
# ConversationBufferMemory: 历史对话消息
from langchain_ollama import ChatOllama
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

llm = ChatOllama(model=os.getenv("MODEL_NAME"), temperature=os.getenv("TEMPERATURE"))
memory = ConversationBufferMemory()

# 创建一个对话，并传入 memory
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)

In [8]:
# 使用 predict 函数开始对话
conversation.predict(input="Hi, my name is Andrew")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my name is Andrew
AI:

> Finished chain.


"Hello Andrew! Nice to meet you. How can I assist you today? Do you have any specific questions or topics you'd like to discuss?"

In [10]:
conversation.predict(input="What is 1+1?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Hello Andrew! Nice to meet you. How can I assist you today? Do you have any specific questions or topics you'd like to discuss?
Human: What is 1+1?
AI:

> Finished chain.


"Hello Andrew! The sum of 1 + 1 is 2. It's a fundamental addition in arithmetic. Is there anything more about math or numbers you're curious about?"

In [11]:
conversation.predict(input="What is my name?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Hello Andrew! Nice to meet you. How can I assist you today? Do you have any specific questions or topics you'd like to discuss?
Human: What is 1+1?
AI: Hello Andrew! The sum of 1 + 1 is 2. It's a fundamental addition in arithmetic. Is there anything more about math or numbers you're curious about?
Human: What is my name?
AI:

> Finished chain.


'Your name is Andrew, Andrew! If you have any questions or would like to discuss anything related to your name or anything else, feel free to ask.'

In [12]:
# 对话的上下文信息
print(memory.buffer)

Human: Hi, my name is Andrew
AI: Hello Andrew! Nice to meet you. How can I assist you today? Do you have any specific questions or topics you'd like to discuss?
Human: What is 1+1?
AI: Hello Andrew! The sum of 1 + 1 is 2. It's a fundamental addition in arithmetic. Is there anything more about math or numbers you're curious about?
Human: What is my name?
AI: Your name is Andrew, Andrew! If you have any questions or would like to discuss anything related to your name or anything else, feel free to ask.


In [14]:
# 获取对话历史
memory.load_memory_variables({})

{'history': "Human: Hi, my name is Andrew\nAI: Hello Andrew! Nice to meet you. How can I assist you today? Do you have any specific questions or topics you'd like to discuss?\nHuman: What is 1+1?\nAI: Hello Andrew! The sum of 1 + 1 is 2. It's a fundamental addition in arithmetic. Is there anything more about math or numbers you're curious about?\nHuman: What is my name?\nAI: Your name is Andrew, Andrew! If you have any questions or would like to discuss anything related to your name or anything else, feel free to ask."}

In [15]:
# 自定义 ConversationBufferMemory: 通过save_context来预设一些对话内容
memory = ConversationBufferMemory()
memory.save_context(
    {"input": "Hi"},
    {"output": "What's up"}
)
print(memory.buffer)

Human: Hi
AI: What's up


In [16]:
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
print(memory.buffer)

Human: Hi
AI: What's up
Human: Not much, just hanging
AI: Cool


In [19]:
# ConversationBufferWindowMemory：以滑动窗口的方式来记录对话内容，可以设置记忆对话的轮数
from langchain_classic.memory import ConversationBufferWindowMemory

# 创建示例，并设置记忆窗口为1，只记住一次对话内容
memory = ConversationBufferWindowMemory(k=1)

memory.save_context({"input": "Hi"},
                    {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
print(memory.buffer)

Human: Not much, just hanging
AI: Cool


In [20]:
# 与LLM结合
llm = ChatOllama(model=os.getenv("MODEL_NAME"), temperature=os.getenv("TEMPERATURE"))
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)

# 多轮对话
conversation.predict(input="Hi, my name is Andrew")
conversation.predict(input="What is 1+1?")
conversation.predict(input="What is my name?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my name is Andrew
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Hello Andrew! Nice to meet you. How are you today? Is there anything interesting or exciting that's been happening in your life lately?
Human: What is 1+1?
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a f

'Hello! Since we just started the conversation and I don\'t have any prior information about you, I don\'t actually know your name yet. You mentioned "Andrew" earlier, but let\'s confirm that. If you tell me your actual name, I can use it in our conversation. How can I help you today?'

In [10]:
# ConversationTokenBufferMemory: Token 的数量来限制记忆的大小
from langchain_classic.memory import ConversationTokenBufferMemory

llm = ChatOllama(model=os.getenv("MODEL_NAME"), temperature=os.getenv("TEMPERATURE"))

# 设置只能记忆 30 个 token 以内的对话
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=20
)

# 构造对话历史
memory.save_context({"input": "AI is what?!"},
                    {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"},
                    {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"},
                    {"output": "Charming!"})

# 打印对话记忆
memory.load_memory_variables({})

{'history': 'AI: Beautiful!\nHuman: Chatbots are what?\nAI: Charming!'}

In [11]:
# ConversationSummaryBufferMemory：将对话内容总结后记忆，把总结的内容作为上下文用于下一次的对话请求
from langchain_classic.memory import ConversationSummaryBufferMemory

# 一段超长的对话内容，超过了设置的 20 最大 Token 数
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=20,
)
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
memory.save_context({"input": "What is on the schedule today?"},
                    {"output": f"{schedule}"})

memory.load_memory_variables({})    # 总结的内容在 system 中，而非之前的 Human 和 AI 字段中

{'history': "System: The human asks about today's schedule, and the AI responds by mentioning a meeting at 8am with the product team and a LangChain project that will progress quickly due to the tool's power. The AI also notes a lunch meeting at an Italian restaurant at noon with a customer interested in learning about the latest advancements in AI, advising to bring a laptop for a demo."}

In [12]:
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)
conversation.predict(input="What would be a good demo to show? Translate to Chinese")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human asks about today's schedule, and the AI responds by mentioning a meeting at 8am with the product team and a LangChain project that will progress quickly due to the tool's power. The AI also notes a lunch meeting at an Italian restaurant at noon with a customer interested in learning about the latest advancements in AI, advising to bring a laptop for a demo.
Human: What would be a good demo to show? Translate to Chinese
AI:

> Finished chain.


'一个合适的产品演示可以展示LangChain技术如何帮助客户解决实际问题，比如快速生成个性化的聊天机器人响应，或者实时处理大量数据以提供洞察。您可以演示一下如何使用该工具来创建一个简单的聊天机器人，并向客户展示其自动回复的能力。\n\n翻译成中文：\n\n一个好的产品演示可以展示 LangChain 技术如何帮助客户解决问题。例如，它可以快速生成个性化的聊天机器人的回应，或实时处理大量数据以提供洞见。您可以演示如何使用该工具创建一个简单的聊天机器人，并向客户展示其自动回复的功能。'

In [13]:
memory.load_memory_variables({})

{'history': "System: The human asks about today's schedule, and the AI responds by mentioning a meeting at 8am with the product team and a LangChain project that will progress quickly due to the tool's power. The AI also notes a lunch meeting at an Italian restaurant at noon with a customer interested in learning about the latest advancements in AI, advising to bring a laptop for a demo.\n\nThe human asks what would be a good demo to show. The AI suggests a product demonstration showcasing how LangChain technology can help solve real-world problems, such as rapidly generating personalized responses from chatbots or processing large amounts of data in real time to provide insights. A simple chatbot creation using the tool and demonstrating its automatic response capabilities could serve as an effective demo.\n\nThe human requests a translation of this suggestion into Chinese:\n\n一个合适的产品演示可以展示 LangChain 技术如何帮助客户解决问题。例如，它可以快速生成个性化的聊天机器人的回应，或实时处理大量数据以提供洞见。您可以演示如何使用该工具创建一个简单的聊天机器人，并向客户展示其自动